In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'retina'

import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

notebook_dir = Path().resolve()
project_root = notebook_dir.parent
src_path = str(project_root / "src")

if src_path not in sys.path:
    sys.path.append(src_path)

from hep_tracking.dataset import TrackDataset
from hep_tracking.features import create_pair_dataset, split_by_event
from hep_tracking.classifiers import (
    RandomForestWrapper, XGBoostWrapper, LightGBMWrapper, 
    evaluate_classifier_throughput, optimize_hyperparameters
)
from hep_tracking.plots import plot_roc_curves, plot_pr_curves
from hep_tracking.utils import calculate_classification_metrics, find_best_f1_threshold

print(f"Katalog główny projektu: {project_root}")
print("Wszystkie moduły załadowane pomyślnie.")

In [ ]:
data_dir = project_root / "data"
dataset_name = "dataset_hard_1M.npz"
candidates_name = "candidates_hard_1M.npz"

data_path = data_dir / dataset_name
candidates_path = data_dir / candidates_name

if not data_path.exists() or not candidates_path.exists():
    raise FileNotFoundError("Brak plików danych. Upewnij się, że wygenerowano dane i kandydatów.")

dataset_raw = TrackDataset.from_npz(data_path)
candidate_indices = np.load(candidates_path)["indices"]

print(f"=== Podsumowanie wczytanych danych surowych ===")
print(f"Plik danych:       {dataset_name}")
print(f"Plik kandydatów:   {candidates_name}")
print(f"Liczba hitów (N):  {len(dataset_raw):,}")
print(f"Wymiar cech hitu:  {dataset_raw.X.shape[1]}")
print(f"Liczba zdarzeń:    {len(np.unique(dataset_raw.event_ids))}")
print(f"Liczba sąsiadów:   {candidate_indices.shape[1]}")

In [ ]:
print("=== Feature Engineering: Tworzenie zbioru par ===")
start_time = time.perf_counter()

dataset_pairs = create_pair_dataset(
    dataset=dataset_raw,
    candidate_indices=candidate_indices,
    max_neg_ratio=5.0,
    seed=42
)

prep_time = time.perf_counter() - start_time

n_total = len(dataset_pairs)
n_pos = np.sum(dataset_pairs.y)
n_neg = n_total - n_pos

print(f"Czas inżynierii cech: {prep_time:.2f} s")
print(f"Liczba par (M):       {n_total:,}")
print(f"Wymiar cech pary:     {dataset_pairs.X.shape[1]}")
print(f"Pary pozytywne:       {n_pos:,} ({n_pos/n_total*100:.1f}%)")
print(f"Pary negatywne:       {n_neg:,} ({n_neg/n_total*100:.1f}%)")

In [ ]:
print("=== Podział na zbiory Train / Val / Test ===")

train_dataset, val_dataset, test_dataset = split_by_event(
    dataset=dataset_pairs,
    train_size=0.75,
    val_size=0.15,
    seed=42
)

print(f"Rozmiar macierzy treningowej:  {len(train_dataset):,} par")
print(f"Rozmiar macierzy walidacyjnej: {len(val_dataset):,} par")
print(f"Rozmiar macierzy testowej:     {len(test_dataset):,} par")
print("Podział zakończony sukcesem. Maski logiczne zapewniają brak przecieku zdarzeń.")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb
from IPython.display import display

print("=== Optymalizacja Hiperparametrów i Trening Modeli ===")

search_size = min(50000, len(train_dataset))
rng = np.random.default_rng(42)
search_indices_train = rng.choice(len(train_dataset), search_size, replace=False)
search_indices_val = rng.choice(len(val_dataset), int(search_size * 0.2), replace=False)

train_sub = TrackDataset(train_dataset.X[search_indices_train], train_dataset.y[search_indices_train], train_dataset.event_ids[search_indices_train])
val_sub = TrackDataset(val_dataset.X[search_indices_val], val_dataset.y[search_indices_val], val_dataset.event_ids[search_indices_val])

trained_models = {}
training_times = {}

print("\n--- Random Forest ---")
rf_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [10, 15, 20, None],
    "min_samples_split": [2, 5, 10]
}
rf_base = RandomForestClassifier(random_state=42)
print("Szukanie hiperparametrów...")
rf_best = optimize_hyperparameters(rf_base, rf_grid, train_sub, val_sub, n_iter=5)
print(f"Najlepsze parametry: {rf_best}")

start_rf = time.perf_counter()
rf_model = RandomForestWrapper(n_jobs=-1, random_state=42, **rf_best)
rf_model.fit(train_dataset.X, train_dataset.y)
trained_models["Random Forest"] = rf_model
training_times["Random Forest"] = time.perf_counter() - start_rf
print(f"Trening zakończony w {training_times['Random Forest']:.2f} s")

print("\n--- XGBoost ---")
xgb_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1]
}
xgb_base = xgb.XGBClassifier(random_state=42, eval_metric="logloss")
print("Szukanie hiperparametrów...")
xgb_best = optimize_hyperparameters(xgb_base, xgb_grid, train_sub, val_sub, n_iter=5)
print(f"Najlepsze parametry: {xgb_best}")

start_xgb = time.perf_counter()
xgb_model = XGBoostWrapper(n_jobs=-1, random_state=42, **xgb_best)
xgb_model.fit(train_dataset.X, train_dataset.y, val_dataset.X, val_dataset.y)
trained_models["XGBoost"] = xgb_model
training_times["XGBoost"] = time.perf_counter() - start_xgb
print(f"Trening zakończony w {training_times['XGBoost']:.2f} s")

print("\n--- LightGBM ---")
lgb_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 6, 8, -1],
    "learning_rate": [0.01, 0.05, 0.1]
}
lgb_base = lgb.LGBMClassifier(random_state=42, verbose=-1)
print("Szukanie hiperparametrów...")
lgb_best = optimize_hyperparameters(lgb_base, lgb_grid, train_sub, val_sub, n_iter=5)
print(f"Najlepsze parametry: {lgb_best}")

start_lgb = time.perf_counter()
lgb_model = LightGBMWrapper(early_stopping_rounds=15, n_jobs=-1, random_state=42, **lgb_best)
lgb_model.fit(train_dataset.X, train_dataset.y, val_dataset.X, val_dataset.y)
trained_models["LightGBM"] = lgb_model
training_times["LightGBM"] = time.perf_counter() - start_lgb
print(f"Trening zakończony w {training_times['LightGBM']:.2f} s")

display(pd.DataFrame(list(training_times.items()), columns=["Model", "Czas treningu [s]"]).round(2))

In [ ]:
lgb_model_instance = trained_models["LightGBM"].model
n_requested = lgb_best["n_estimators"]
n_actual = lgb_model_instance.booster_.best_iteration

print(f"LightGBM: zażądano {n_requested} drzew, early stopping zatrzymał trening na {n_actual} drzewie")
if n_actual < n_requested:
    print(f"Early stopping faktycznie zadziałał — oszczędność {n_requested - n_actual} drzew ({(1 - n_actual/n_requested):.0%})")
else:
    print("Early stopping nie zatrzymał treningu wcześniej — walidacyjny ROC-AUC rósł przez wszystkie zadane rundy")

In [ ]:
print("=== Optymalny próg decyzyjny (maksymalizujący F1) ===")

for model_name, model in trained_models.items():
    y_pred_proba = model.predict_proba(test_dataset.X)[:, 1]
    best_threshold, best_f1 = find_best_f1_threshold(test_dataset.y, y_pred_proba)
    metrics_at_default = calculate_classification_metrics(test_dataset.y, y_pred_proba, threshold=0.5)
    print(
        f"{model_name}: próg=0.50 -> F1={metrics_at_default['F1-Score']:.4f}  |  "
        f"optymalny próg={best_threshold:.3f} -> F1={best_f1:.4f}"
    )

In [ ]:
print("=== Ewaluacja jakości klasyfikacji (Zbiór Testowy) ===")

metrics_list = []

for model_name, model in trained_models.items():
    y_pred_proba = model.predict_proba(test_dataset.X)[:, 1]
    
    metrics = calculate_classification_metrics(test_dataset.y, y_pred_proba)
    metrics["Model"] = model_name
    
    metrics_list.append(metrics)

df_metrics = pd.DataFrame(metrics_list)
df_metrics.set_index("Model", inplace=True)

display(df_metrics.round(4))

In [ ]:
print("=== Wykresy (Deliverables) ===")

plot_roc_curves(trained_models, test_dataset)
plot_pr_curves(trained_models, test_dataset)

In [ ]:
print("=== Analiza przepustowości (Throughput / Latency) ===")

throughput_results = []
batch_sizes_to_test = [1, 1000, 10000]

for model_name, model in trained_models.items():
    results = evaluate_classifier_throughput(
        model=model, 
        test_dataset=test_dataset, 
        batch_sizes=batch_sizes_to_test, 
        num_runs=5
    )
    
    for batch_size, throughput in results.items():
        throughput_results.append({
            "Model": model_name,
            "Rozmiar wsadu (Batch)": batch_size,
            "Przepustowość (pary/s)": throughput
        })

df_throughput = pd.DataFrame(throughput_results)

df_pivot = df_throughput.pivot(
    index="Model", 
    columns="Rozmiar wsadu (Batch)", 
    values="Przepustowość (pary/s)"
)

df_pivot_formatted = df_pivot.map(lambda x: f"{x:,.0f}")
display(df_pivot_formatted)

In [ ]:
print("=== Eksperyment kontrolny: kNN kandydaci tylko na współrzędnych przestrzennych ===")
from hep_tracking.generate_candidates import generate_and_save_candidates

generate_and_save_candidates(data_dir=str(data_dir), spatial_only=True)

candidates_spatial_path = data_dir / "candidates_hard_1M_spatial.npz"
candidate_indices_spatial = np.load(candidates_spatial_path)["indices"]

dataset_pairs_sp = create_pair_dataset(
    dataset=dataset_raw,
    candidate_indices=candidate_indices_spatial,
    max_neg_ratio=5.0,
    seed=42,
)

train_dataset_sp, val_dataset_sp, test_dataset_sp = split_by_event(
    dataset=dataset_pairs_sp,
    train_size=0.75, val_size=0.15, seed=42,
)

rf_spatial = RandomForestWrapper(n_jobs=-1, random_state=42, **rf_best)
rf_spatial.fit(train_dataset_sp.X, train_dataset_sp.y)

y_pred_proba_sp = rf_spatial.predict_proba(test_dataset_sp.X)[:, 1]
metrics_spatial = calculate_classification_metrics(test_dataset_sp.y, y_pred_proba_sp)

print(f"Pary pozytywne (spatial-only kandydaci): {dataset_pairs_sp.y.mean():.1%}")
print(f"Random Forest, kandydaci PEŁNI (x,y,z,dir):     ROC-AUC={df_metrics.loc['Random Forest', 'ROC-AUC']:.4f}")
print(f"Random Forest, kandydaci SPATIAL-ONLY (x,y,z):  ROC-AUC={metrics_spatial['ROC-AUC']:.4f}, F1={metrics_spatial['F1-Score']:.4f}")

if metrics_spatial["ROC-AUC"] < 0.999:
    print("\n>>> Hipoteza POTWIERDZONA: bez sygnału kierunkowego w kNN klasyfikacja przestaje być idealna.")
else:
    print("\n>>> Hipoteza NIE potwierdzona: RandomForest nadal niemal idealny nawet bez sygnału kierunkowego w kNN — separowalność pochodzi z samej geometrii danych, nie z metody doboru kandydatów.")

In [ ]:
lgb_saved_pct = (1 - n_actual / n_requested) if n_actual < n_requested else 0

lgb_throughput_rows = df_throughput[df_throughput["Model"] == "LightGBM"]
max_batch_col = lgb_throughput_rows["Rozmiar wsadu (Batch)"].max()
lgb_throughput_max_batch = lgb_throughput_rows.loc[
    lgb_throughput_rows["Rozmiar wsadu (Batch)"] == max_batch_col,
    "Przepustowość (pary/s)"
].values[0]

summary = f"""
## Podsumowanie

**Wybór modelu:** {df_metrics['ROC-AUC'].idxmax()} ma najwyższy ROC-AUC ({df_metrics['ROC-AUC'].max():.4f}).
LightGBM ma najwyższą przepustowość wsadową przy batch={max_batch_col}
({lgb_throughput_max_batch:,.0f} par/s), co czyni go najlepszym kandydatem
do szybkiego pipeline'u.

**Early stopping LightGBM:** zażądano {n_requested} drzew, zatrzymano na {n_actual}
({lgb_saved_pct:.0%} oszczędności).

**Random Forest = {df_metrics.loc['Random Forest', 'ROC-AUC']:.4f} ROC-AUC na teście:**
{"Kontrolny eksperyment ze spatial-only kandydatami POTWIERDZIŁ" if metrics_spatial["ROC-AUC"] < 0.999 else "Kontrolny eksperyment ze spatial-only kandydatami NIE potwierdził"}
hipotezę o wycieku sygnału kierunkowego przez sposób doboru kandydatów kNN
(spatial-only ROC-AUC = {metrics_spatial['ROC-AUC']:.4f}).

**Stosunek negatywy:pozytywy:** {(1 - dataset_pairs.y.mean())/dataset_pairs.y.mean():.2f}:1 —
poniżej zadeklarowanego limitu 5:1 (limit górny, nie sztywny cel).
"""

from IPython.display import Markdown, display
display(Markdown(summary))